In [5]:
!pip install -q langchain langchain-openai
from google.colab import userdata
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=0.3
)
print("LangChain + OpenAI-compatible Groq setup successful!")

LangChain + OpenAI-compatible Groq setup successful!


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()
history = []
def ask(question: str) -> str:
  answer = chain.invoke({
        "history": history,
        "question": question
    })
  history.extend([
        ("human", question),
        ("ai", answer)
    ])
  return answer
print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Yes, all our cakes have eggless versions.
I'll check with the owner.


In [9]:
import csv
def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")
run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500.
FAIL | Do you make eggless cakes? → Yes, we offer egg‑less versions of all our cakes—plum, chocolate, and 
FAIL | Can I get a cake today evening? → I’m sorry, we need a 24‑hour notice for orders. I’ll check with the ow
PASS | Do you deliver to Kakinada? → I’ll check with the owner and get back to you.
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480. Let me know if you’d like to place an order!

3/5 passed (60%)


In [10]:
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})
print(result["messages"][-1].content)

Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Feel free to let us know your preference!


In [14]:
#I choosen Mobile Shop using  AI Assistant using LangChain + Groq.

In [15]:
SYSTEM = """ROLE:
You are a friendly AI assistant for Smart Mobile Store.

FACTS:
iPhone 15 price Rs 55000.
Samsung Galaxy S24 price Rs 45000.
OnePlus 12 price Rs 60000.
Redmi Note 13 price Rs 18000.
Orders need 24 hours' notice.

RULES:
Never invent prices or product details.
If information is unavailable, say "I'll check with the owner."
Calculate totals carefully.
Never promise delivery time or place.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer.
"""

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()
history = []

In [17]:
print(ask("What is the price of iPhone 15?"))
print(ask("What is the price of Samsung Galaxy S24?"))
print(ask("Do you have Redmi Note 13?"))

Hi! The iPhone 15 is priced at ₹55,000. Let me know if you’d like to explore other models or place an order!
Hi! The Samsung Galaxy S24 costs ₹45,000. Let me know if you’d like to compare it with other phones or place an order!
Hi! The Redmi Note 13 is ₹18,000. I’m not sure about current stock—I'll check with the owner and get back to you.


In [25]:
import os

print("Current folder:", os.getcwd())
print("CSV file exists:", os.path.exists("tests.csv"))

print("\nActual tests.csv content:\n")

with open("tests.csv", "r") as f:
    print(f.read())

Current folder: /content
CSV file exists: True

Actual tests.csv content:

question,must_include
"What is the price of iPhone 15?","55000"
"What is the price of Samsung Galaxy S24?","45000"
"What is the price of OnePlus 12?","60000"
"What is the price of Redmi Note 13?","18000"
"Do you have iPhone 15?","55000"
"Can I order a mobile today?","24 hours"
"How much does OnePlus 12 cost?","60000"
"What is the price of Redmi Note 13?","18000"
"Do you deliver to Kakinada?","check with the owner"
"Can I get a discount on iPhone 15?","check with the owner"



In [28]:
import csv
def run_tests(path="tests.csv"):
    with open(path, "r") as file:
        rows = list(csv.DictReader(file))
    passed = 0
    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        if ok:
            passed += 1
        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "→",
            answer[:70]
        )
    total = len(rows)
    print(f"\n{passed}/{total} passed ({passed/total:.0%})")
run_tests()

FAIL | What is the price of iPhone 15? → Hi! The iPhone 15 is priced at ₹55,000. Let me know if you’d like to a
FAIL | What is the price of Samsung Galaxy S24? → Samsung Galaxy S24 costs ₹45,000.
FAIL | What is the price of OnePlus 12? → The OnePlus 12 is priced at ₹60,000.
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000.
FAIL | Do you have iPhone 15? → Yes, we have the iPhone 15 for Rs 55,000. Please give a 24‑hour notice
FAIL | Can I order a mobile today? → Sure! I can take your order, but we need a 24‑hour notice before placi
FAIL | How much does OnePlus 12 cost? → OnePlus 12 costs ₹60,000. Let me know if you’d like to place an order 
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000.
PASS | Do you deliver to Kakinada? → I’ll check with the owner and let you know.
FAIL | Can I get a discount on iPhone 15? → Sorry, the iPhone 15 is priced at ₹55,000 and we don’t offer discounts

1/10 passed (10%)


In [ ]:
#Improve the SYSTEM prompt
#Objective: Improve your assistant's responses and compare the test results before and after the improvement.

In [30]:
SYSTEM = """ROLE:
You are a friendly and helpful AI assistant for Smart Mobile Store.

FACTS:
iPhone 15 price Rs 55000.
Samsung Galaxy S24 price Rs 45000.
OnePlus 12 price Rs 60000.
Redmi Note 13 price Rs 18000.
Orders need 24 hours' notice.

RULES:
1. Always answer using the given FACTS.
2. When a customer asks for a product price, mention the exact price.
3. Never invent prices or product details.
4. If information is unavailable, say "I'll check with the owner."
5. Never promise delivery time or place.
6. Calculate totals carefully.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer.
"""

In [29]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

In [31]:
run_tests()

FAIL | What is the price of iPhone 15? → The iPhone 15 is priced at ₹55,000.
FAIL | What is the price of Samsung Galaxy S24? → The Samsung Galaxy S24 costs ₹45,000.
FAIL | What is the price of OnePlus 12? → OnePlus 12 costs ₹60,000.
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000.
FAIL | Do you have iPhone 15? → Yes, we have the iPhone 15! It’s priced at Rs 55,000. Just let us know
FAIL | Can I order a mobile today? → Sure! I can take your order, but we need a 24‑hour notice. Which phone
FAIL | How much does OnePlus 12 cost? → The OnePlus 12 costs ₹60,000. If you’d like to place an order, let me 
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000. Let me know if you’d like to a
PASS | Do you deliver to Kakinada? → I’ll check with the owner and get back to you.
PASS | Can I get a discount on iPhone 15? → Sorry, no discount is available on the iPhone 15 at the moment. If you

2/10 passed (20%)


In [33]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=1.0
)

print("Temperature changed to 1.0!")

Temperature changed to 1.0!


In [34]:
chain = prompt | llm | StrOutputParser()

In [35]:
print("First Run:")
run_tests()

First Run:
FAIL | What is the price of iPhone 15? → The iPhone 15 is priced at ₹55,000.
FAIL | What is the price of Samsung Galaxy S24? → Samsung Galaxy S24 costs Rs 45,000.
FAIL | What is the price of OnePlus 12? → The OnePlus 12 is priced at ₹60,000. Let me know if you’d like to expl
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000.
FAIL | Do you have iPhone 15? → Yes, we have the iPhone 15. It’s ₹55,000. Please place your order at l
FAIL | Can I order a mobile today? → Sure! To place an order, we need a 24‑hour notice. For example, iPhone
FAIL | How much does OnePlus 12 cost? → The OnePlus 12 costs Rs 60,000.
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at **Rs 18,000**.
PASS | Do you deliver to Kakinada? → Sure! I’ll check with the owner to confirm delivery to Kakinada. Thank
PASS | Can I get a discount on iPhone 15? → I’m sorry, but we don’t offer a discount on the iPhone 15, priced at R

2/10 passed (20%)


In [36]:
print("Second Run:")
run_tests()

Second Run:
FAIL | What is the price of iPhone 15? → The iPhone 15 is priced at **Rs 55,000**. Let me know if you’d like to
FAIL | What is the price of Samsung Galaxy S24? → Samsung Galaxy S24 is ₹45,000.
FAIL | What is the price of OnePlus 12? → The OnePlus 12 costs **₹60,000**. Let me know if you’d like to place a
FAIL | What is the price of Redmi Note 13? → The Redmi Note 13 is priced at ₹18,000.
FAIL | Do you have iPhone 15? → Yes we have iPhone 15, priced at ₹55,000. To place an order, please gi
FAIL | Can I order a mobile today? → I’m sorry, but orders need a 24‑hour notice. I can note your request a
FAIL | How much does OnePlus 12 cost? → OnePlus 12 costs ₹60,000.  To place an order, please give us a 24‑hour
FAIL | What is the price of Redmi Note 13? → Sure! The Redmi Note 13 is priced at **₹18,000**. Feel free to let me 
PASS | Do you deliver to Kakinada? → I’m not sure about delivery to Kakinada. I’ll check with the owner.
FAIL | Can I get a discount on iPhone 15? → Sorry, we 

In [37]:
jobs = [
    {
        "job": "Calculate the total price of 3 mobile phones",
        "type": "Plain Code"
    },
    {
        "job": "Answer customer questions about mobile features",
        "type": "One LLM Call"
    },
    {
        "job": "Check mobile stock availability using a tool",
        "type": "Agent"
    }
]
for item in jobs:
    print("Job:", item["job"])
    print("Type:", item["type"])
    print()

Job: Calculate the total price of 3 mobile phones
Type: Plain Code

Job: Answer customer questions about mobile features
Type: One LLM Call

Job: Check mobile stock availability using a tool
Type: Agent

